# Customer Churn Prediction
## Notebook 2 of 8 — Data Cleaning

This project predicts which telecom customers are likely to **churn** (cancel their service) so the business can reach them with retention offers *before* they leave. It walks through the full data-science lifecycle — exploration, cleaning, EDA, feature engineering, modelling, evaluation, and a tuned final model.

**Business problem:** Winning a new customer costs far more than keeping an existing one. This telecom loses roughly **27% of its customers**, and the leadership team wants a reliable, data-driven way to flag at-risk customers early enough to act.

**Tools & techniques:** Python · pandas · NumPy · Matplotlib · Seaborn · scikit-learn · XGBoost · SMOTE (imbalanced-learn) · joblib

> **This notebook:** the single source of truth for cleaning — we fix the `TotalCharges` data type, handle the hidden blanks, check for duplicates, and save a clean dataset that every later notebook reuses.

**Author:** [Your Name]  &nbsp;·&nbsp;  **Last updated:** June 2026

---

## 1. Load the raw data

**What:** Read the original dataset again, untouched.

**Why:** Cleaning should always start from the raw source so the steps are fully reproducible. This notebook is the **only** place cleaning happens — downstream notebooks load the cleaned output instead of repeating this logic.

In [1]:
# Standard library
import warnings

# Third-party
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

df = pd.read_csv("../data/raw/telco-customer-churn.csv")
print(f"Loaded raw data: {df.shape[0]} rows, {df.shape[1]} columns")

C:\Users\Vivobook\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Loaded raw data: 7043 rows, 21 columns


## 2. Fix `TotalCharges`

**What:** Replace the 11 blank `TotalCharges` values with `0`, then convert the whole column to a numeric (float) type.

**Why:** As we saw in Notebook 01, the blanks all belong to customers with `tenure = 0` — brand-new customers who have not been billed yet, so **0 is the correct, defensible fill value** (not a guess or an average). Converting to float lets models and charts treat it as a real number.

In [2]:
# Blank TotalCharges = brand-new customers (tenure 0) -> bill of 0
df["TotalCharges"] = df["TotalCharges"].replace(" ", "0").astype(float)

# Confirm the column is now numeric and no blanks remain
print("Dtype:", df["TotalCharges"].dtype)
print("Blank values left:", (df["TotalCharges"].astype(str) == " ").sum())

Dtype: float64
Blank values left: 0


`TotalCharges` is now a clean **float64** column with **zero** blank values — ready for analysis and modelling.

## 3. Check for duplicate rows

**What:** Count exact duplicate customer records.

**Why:** Duplicates would over-weight some customers and leak between train and test sets, quietly inflating model scores.

In [3]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 0


**No duplicate rows** — every record is a unique customer, so no de-duplication is needed.

## 4. Save the cleaned dataset

**What:** Write the cleaned, still-human-readable data to `data/processed/telco_churn_clean.csv`.

**Why:** This is the **single source of truth** for the rest of the project. The EDA and feature-engineering notebooks load this file instead of re-cleaning the raw data, which keeps the pipeline consistent and avoids copy-pasted cleaning logic.

Note: we keep the original categorical columns here (no encoding yet) so the EDA charts stay readable. Encoding happens in Notebook 04.

In [4]:
df.to_csv("../data/processed/telco_churn_clean.csv", index=False)
print("Saved -> data/processed/telco_churn_clean.csv")
print(f"Shape: {df.shape}")

Saved -> data/processed/telco_churn_clean.csv
Shape: (7043, 21)


## Section conclusion — what cleaning achieved

- Converted **`TotalCharges` from text to float**, fixing the data type that would otherwise break modelling.
- Filled the **11 blank values with 0**, the correct value for `tenure = 0` new customers.
- Confirmed there are **no duplicate records**.
- Saved a tidy, human-readable dataset as the single source of truth for every downstream notebook.

**Next:** Notebook 03 uses this clean data to explore *which* kinds of customers churn.